# ReLLa-lite QLoRA continuation: +10k train examples, no evaluation

This notebook continues training the existing ReLLa-lite QLoRA adapter.

Goal:

```text
ReLLa-lite 10k adapter
+ 10k new disjoint train examples
= ReLLa-lite adapter exposed to 20k train examples in total
```

Validation/test evaluation is handled in a separate eval-only notebook after the new adapter is saved.

## Input artifacts

### 1. Retrieved instruction dataset

Expected files:

```text
train.jsonl
val.jsonl
test.jsonl
```

The dataset corresponds to `instruction_temporal_retrieved_topk`. Only `train.jsonl` is required for this training-only run.

### 2. Existing ReLLa-lite 10k adapter

Expected adapter files:

```text
adapter_qwen_rella_lite/
  adapter_config.json
  adapter_model.safetensors
  tokenizer.json
  tokenizer_config.json
  README.md
```

The required files for continuation training are `adapter_config.json` and `adapter_model.safetensors`.


In [ ]:
!pip install -q -U "transformers>=4.45.0" accelerate peft bitsandbytes scikit-learn pandas pyarrow safetensors


In [ ]:
import os, json, glob, math, random, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import PeftModel, prepare_model_for_kbit_training

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("device count:", torch.cuda.device_count())


## 1. Configuration

The previous ReLLa-lite run used `PREVIOUS_PER_CLASS = 5000`. This notebook adds `NEW_PER_CLASS = 5000`, so the adapter will see 10k old + 10k new examples in total.


In [ ]:
MODEL_NAME_FALLBACK = "Qwen/Qwen3-4B-Instruct-2507"
MAX_SEQ_LENGTH = 1024
PREVIOUS_PER_CLASS = 5000
NEW_PER_CLASS = 5000
NUM_EPOCHS = 1
LR = 1e-4
GRAD_ACCUM = 8
BATCH_SIZE = 1
SAVE_STEPS = 500

RUN_NAME = f"qwen_rella_lite_continue_new{NEW_PER_CLASS*2}_from10k_no_eval"
OUT_DIR = Path("/kaggle/working") / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_NAME:", RUN_NAME)
print("OUT_DIR:", OUT_DIR)


## 2. Locate train.jsonl and adapter


In [ ]:
def find_file(filename, prefer_keywords=None, avoid_keywords=None):
    hits = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"Required file was not found in /kaggle/input: {filename}")
    prefer_keywords = [x.lower() for x in (prefer_keywords or [])]
    avoid_keywords = [x.lower() for x in (avoid_keywords or [])]
    def score(path):
        p = path.lower()
        prefer_penalty = sum(kw not in p for kw in prefer_keywords)
        avoid_penalty = sum(kw in p for kw in avoid_keywords)
        return (prefer_penalty, avoid_penalty, len(path))
    return Path(sorted(hits, key=score)[0])

train_path = find_file("train.jsonl", prefer_keywords=["retrieved", "topk", "instruction"])
adapter_config_path = find_file(
    "adapter_config.json",
    prefer_keywords=["rella", "adapter", "train10000"],
    avoid_keywords=["ordinary", "checkpoint3000", "checkpoint2500"]
)
adapter_dir = adapter_config_path.parent

print("train_path:", train_path)
print("adapter_dir:", adapter_dir)
print("adapter files:", [p.name for p in adapter_dir.iterdir() if p.is_file()][:30])


In [ ]:
with open(adapter_config_path, "r", encoding="utf-8") as f:
    adapter_cfg = json.load(f)

MODEL_NAME = adapter_cfg.get("base_model_name_or_path") or MODEL_NAME_FALLBACK
if MODEL_NAME.startswith("/") or MODEL_NAME.startswith("./"):
    MODEL_NAME = MODEL_NAME_FALLBACK

print("MODEL_NAME:", MODEL_NAME)
print("adapter config subset:")
print(json.dumps({
    "base_model_name_or_path": adapter_cfg.get("base_model_name_or_path"),
    "peft_type": adapter_cfg.get("peft_type"),
    "task_type": adapter_cfg.get("task_type"),
    "r": adapter_cfg.get("r"),
    "lora_alpha": adapter_cfg.get("lora_alpha"),
    "target_modules": adapter_cfg.get("target_modules"),
}, ensure_ascii=False, indent=2))


## 3. Load data and select a disjoint +10k chunk


In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = load_jsonl(train_path)
print("train shape:", train_df.shape)
print("output distribution:", train_df["output"].value_counts(dropna=False).to_dict())

def class_mask(df, cls):
    return df["output"].astype(str).str.strip().str.lower() == cls.lower()

def make_continuation_chunk(df, previous_per_class, new_per_class, seed=42):
    yes = df[class_mask(df, "yes")]
    no = df[class_mask(df, "no")]
    if len(yes) < previous_per_class + new_per_class:
        raise ValueError("Not enough Yes examples for the requested continuation chunk.")
    if len(no) < previous_per_class + new_per_class:
        raise ValueError("Not enough No examples for the requested continuation chunk.")
    previous_yes_idx = yes.sample(n=previous_per_class, random_state=seed).index
    previous_no_idx = no.sample(n=previous_per_class, random_state=seed).index
    yes_remaining = yes.drop(index=previous_yes_idx)
    no_remaining = no.drop(index=previous_no_idx)
    new_yes = yes_remaining.sample(n=new_per_class, random_state=seed + 100)
    new_no = no_remaining.sample(n=new_per_class, random_state=seed + 100)
    chunk = pd.concat([new_yes, new_no], axis=0)
    chunk = chunk.sample(frac=1.0, random_state=seed + 101).reset_index(drop=True)
    diagnostics = {
        "previous_yes": int(len(previous_yes_idx)),
        "previous_no": int(len(previous_no_idx)),
        "new_yes": int(len(new_yes)),
        "new_no": int(len(new_no)),
        "remaining_yes_after_old": int(len(yes_remaining)),
        "remaining_no_after_old": int(len(no_remaining)),
    }
    if "sample_id" in df.columns:
        previous_ids = set(df.loc[list(previous_yes_idx) + list(previous_no_idx), "sample_id"].tolist())
        new_ids = set(chunk["sample_id"].tolist())
        diagnostics["sample_id_overlap_with_previous"] = int(len(previous_ids.intersection(new_ids)))
    return chunk, diagnostics

train_chunk_df, chunk_diagnostics = make_continuation_chunk(train_df, PREVIOUS_PER_CLASS, NEW_PER_CLASS, SEED)
print("train chunk:", train_chunk_df.shape)
print("chunk output distribution:", train_chunk_df["output"].value_counts().to_dict())
print("chunk diagnostics:", chunk_diagnostics)


## 4. Tokenizer and dataset


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

SYSTEM_PROMPT = "You are a recommendation model. Answer only Yes or No."

def normalize_answer(x):
    x = str(x).strip()
    if x.lower().startswith("yes"):
        return "Yes"
    if x.lower().startswith("no"):
        return "No"
    raise ValueError(f"Unexpected answer value: {x}")

def build_user_text(row):
    instruction = str(row.get("instruction", "")).strip()
    inp = str(row.get("input", "")).strip()
    if instruction and inp:
        return instruction + "\n\n" + inp
    if instruction:
        return instruction
    if "prompt_text" in row and pd.notna(row["prompt_text"]):
        return str(row["prompt_text"]).strip()
    raise ValueError("The row does not contain instruction/input fields required for prompt construction.")

def make_prompt_text(row):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_text(row)}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def make_answer_text(row):
    return normalize_answer(row["output"]) + tokenizer.eos_token

print(make_prompt_text(train_chunk_df.iloc[0])[:2500])
print("answer:", make_answer_text(train_chunk_df.iloc[0]))


In [ ]:
class RecSFTDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_length=1024):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_ids = self.tokenizer(make_prompt_text(row), add_special_tokens=False).input_ids
        answer_ids = self.tokenizer(make_answer_text(row), add_special_tokens=False).input_ids
        max_prompt_len = self.max_length - len(answer_ids)
        if len(prompt_ids) > max_prompt_len:
            prompt_ids = prompt_ids[-max_prompt_len:]
        input_ids = prompt_ids + answer_ids
        labels = [-100] * len(prompt_ids) + answer_ids
        attention_mask = [1] * len(input_ids)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_mask, labels = [], [], []
    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        input_ids.append(torch.cat([x["input_ids"], torch.full((pad_len,), tokenizer.pad_token_id, dtype=torch.long)]))
        attention_mask.append(torch.cat([x["attention_mask"], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(torch.cat([x["labels"], torch.full((pad_len,), -100, dtype=torch.long)]))
    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_mask),
        "labels": torch.stack(labels),
    }

train_ds = RecSFTDataset(train_chunk_df, tokenizer, MAX_SEQ_LENGTH)
lens = [len(train_ds[i]["input_ids"]) for i in range(min(1000, len(train_ds)))]
print("length min/mean/max:", min(lens), float(np.mean(lens)), max(lens))
print("train examples:", len(train_ds))


## 5. Load base model and trainable existing adapter


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)

model = PeftModel.from_pretrained(base_model, str(adapter_dir), is_trainable=True)
model.print_trainable_parameters()


## 6. Continue training and save adapter immediately


In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUT_DIR / "trainer"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    fp16=True,
    logging_steps=20,
    save_steps=SAVE_STEPS,
    save_strategy="steps",
    save_total_limit=3,
    report_to="none",
    optim="paged_adamw_8bit",
    seed=SEED,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    warmup_ratio=0.03,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collate_fn,
)

print("approx optimization steps:", math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS)
trainer.train()

adapter_out_dir = OUT_DIR / "adapter_qwen_rella_lite_continued_20k"
model.save_pretrained(adapter_out_dir)
tokenizer.save_pretrained(adapter_out_dir)
shutil.make_archive(str(OUT_DIR / "adapter_qwen_rella_lite_continued_20k"), "zip", adapter_out_dir)

run_summary = {
    "model_name": MODEL_NAME,
    "source_adapter_dir": str(adapter_dir),
    "source_dataset": str(train_path),
    "dataset": "instruction_temporal_retrieved_topk",
    "previous_per_class": int(PREVIOUS_PER_CLASS),
    "new_per_class": int(NEW_PER_CLASS),
    "new_train_examples": int(len(train_chunk_df)),
    "total_exposed_examples_estimate": int(2 * PREVIOUS_PER_CLASS + 2 * NEW_PER_CLASS),
    "num_epochs_on_new_chunk": int(NUM_EPOCHS),
    "learning_rate": float(LR),
    "max_seq_length": int(MAX_SEQ_LENGTH),
    "chunk_diagnostics": chunk_diagnostics,
    "output_adapter_dir": str(adapter_out_dir),
}
with open(OUT_DIR / "qwen_rella_lite_continued_20k_summary.json", "w", encoding="utf-8") as f:
    json.dump(run_summary, f, ensure_ascii=False, indent=2)

print("saved:", OUT_DIR)
print("adapter zip:", OUT_DIR / "adapter_qwen_rella_lite_continued_20k.zip")
print("summary:", OUT_DIR / "qwen_rella_lite_continued_20k_summary.json")


## Output artifacts

Main artifacts:

```text
adapter_qwen_rella_lite_continued_20k.zip
qwen_rella_lite_continued_20k_summary.json
```

Evaluation input directory:

```text
adapter_qwen_rella_lite_continued_20k/
  adapter_config.json
  adapter_model.safetensors
  tokenizer.json
  tokenizer_config.json
  README.md
```

The final saved adapter directory is the evaluation input.
